# Faruq-v3 AF2 — reused-test post-hoc

Mengevaluasi AF2 seed 42/123/2026 pada paket Faruq test yang sebelumnya sudah dipakai studi ACMC. Ini **bukan locked-test confirmation baru**. D0FT tidak dievaluasi ulang; tiga report historisnya dipakai hanya bila SHA manifest test sama. Tidak ada training, pemilihan checkpoint, atau tuning setelah hasil.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/af2-continuation-confirmation'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.96', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('REPO:', REPO, '| BRANCH:', BRANCH)

In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

ARCHIVE_REL = 'bundles/faruq-v3-locked-test-v1.tar'
D0FT_REL = (
    'experiments/faruq-v3-acmc-locked-test-v2/reports/D0FT_seed42_test.json',
    'experiments/faruq-v3-acmc-locked-test-v2/reports/D0FT_seed123_test.json',
    'experiments/faruq-v3-acmc-locked-test-v2/reports/D0FT_seed2026_test.json',
)
AF2_REL = (
    'experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt',
    'experiments/faruq-v3-af2-igem-paired-confirmation-v1/AF2/AF2_seed123/weights/best.pt',
    'experiments/faruq-v3-af2-igem-paired-confirmation-v1/AF2/AF2_seed2026/weights/best.pt',
)
REQUIRED = (ARCHIVE_REL, *D0FT_REL, *AF2_REL)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=REQUIRED)
ARCHIVE = require_project_artifact(PROJECT_ROOT, ARCHIVE_REL)
D0FT_REPORTS = tuple(require_project_artifact(PROJECT_ROOT, path) for path in D0FT_REL)
AF2_CHECKPOINTS = tuple(require_project_artifact(PROJECT_ROOT, path) for path in AF2_REL)
TEST_ROOT = Path('/content/faruq-v3-locked-test')
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-af2-reused-test-posthoc-v1'
if TEST_ROOT.exists():
    shutil.rmtree(TEST_ROOT)
with tarfile.open(ARCHIVE, 'r') as archive:
    archive.extractall('/content', filter='data')
assert (TEST_ROOT / 'data.yaml').is_file(), TEST_ROOT
assert (TEST_ROOT / 'faruq_locked_test_manifest.json').is_file(), TEST_ROOT
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('PROJECT:', PROJECT_ROOT)
print('TEST   :', TEST_ROOT)
print('OUTPUT :', OUTPUT_ROOT)
print('D0FT reports:', [str(path) for path in D0FT_REPORTS])
print('AF2 checkpoints:', [str(path) for path in AF2_CHECKPOINTS])

In [ ]:
import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_af2_reused_test',
    '--test-root', str(TEST_ROOT),
    '--d0ft-reports', *map(str, D0FT_REPORTS),
    '--af2-checkpoints', *map(str, AF2_CHECKPOINTS),
    '--output-root', str(OUTPUT_ROOT),
    '--device', '0', '--authorize-reused-test',
]
LOG_PATH = OUTPUT_ROOT / 'af2_reused_test_posthoc_run.log'
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with LOG_PATH.open('a', encoding='utf-8') as log:
    for line in process.stdout:
        log.write(line); log.flush()
        if line.startswith('REUSED TEST') or line.startswith('BOOTSTRAP') or line.startswith('SAVED:'):
            print(line, end='', flush=True)
return_code = process.wait()
if return_code != 0:
    tail = LOG_PATH.read_text(encoding='utf-8', errors='replace').splitlines()[-120:]
    print('\n=== 120 BARIS TERAKHIR ===')
    print('\n'.join(tail))
    raise RuntimeError(f'Evaluasi gagal dengan return code {return_code}. Log: {LOG_PATH}')
print('SELESAI. LOG:', LOG_PATH)

In [ ]:
import pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'af2_reused_test_posthoc_summary.json'
assert SUMMARY.is_file(), SUMMARY
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['training_executed'] is False
assert result['test_images_accessed'] is True
assert result['scientific_status'] == 'REUSED_TEST_POSTHOC_NOT_LOCKED_CONFIRMATION'
rows = [{'metric': metric, **values} for metric, values in result['aggregate'].items()]
columns = ('d0ft_mean', 'd0ft_std', 'af2_mean', 'af2_std', 'delta_mean', 'delta_std', 'delta_min')
display(pd.DataFrame(rows).style.format({column: '{:.2%}' for column in columns}))
print('BOOTSTRAP:', json.dumps(result['paired_parent_bootstrap'], indent=2))
print('STATUS:', result['status'])
print('SCIENTIFIC STATUS:', result['scientific_status'])
print('SUMMARY:', SUMMARY)
print('Kirim tabel, bootstrap, dan status. Ini reused-test post-hoc; jangan tuning.')